# Review status: experimental draft — do not treat as validated trajectory prediction

This notebook was statically reviewed on 2026-10-14 but has not been executed end-to-end. Its pseudo `object_id`/`frame_id` construction groups every five rows and does not represent verified temporal tracks of the same physical object. The resulting next-position targets and threshold metrics are exploratory only. Verify dataset provenance/license, class mapping, image extensions, paths, and evaluation before reporting any results.


In [ ]:
import os
import pandas as pd

# -------------------------------
# 1. Set dataset paths
# -------------------------------
BASE_DIR = '/kaggle/input/extract'

TRAIN_IMAGES_DIR = os.path.join(BASE_DIR, 'train', 'images')
TRAIN_LABELS_DIR = os.path.join(BASE_DIR, 'train', 'labels')

VALID_IMAGES_DIR = os.path.join(BASE_DIR, 'valid', 'images')
VALID_LABELS_DIR = os.path.join(BASE_DIR, 'valid', 'labels')

# -------------------------------
# 2. Define class names manually
# -------------------------------
class_names = ['debris', 'satellite']  # Replace with your actual class names

# -------------------------------
# 3. Function to parse YOLO labels and convert to CSV rows
# -------------------------------
def parse_labels(labels_dir, images_dir, split_name):
    rows = []
    if not os.path.exists(labels_dir):
        print(f"Warning: Labels directory does not exist: {labels_dir}")
        return rows

    for label_file in os.listdir(labels_dir):
        if not label_file.endswith('.txt'):
            continue
        
        label_path = os.path.join(labels_dir, label_file)
        image_filename = label_file.replace('.txt', '.jpg')  # adjust if images are .png
        image_path = os.path.join(images_dir, image_filename)

        if not os.path.exists(image_path):
            print(f"Warning: Image not found for label {label_file}")
            continue

        with open(label_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                class_id, x_center, y_center, width, height = parts
                rows.append({
                    'split': split_name,
                    'image_path': image_path,
                    'class_id': int(class_id),
                    'class_name': class_names[int(class_id)],
                    'x_center': float(x_center),
                    'y_center': float(y_center),
                    'width': float(width),
                    'height': float(height)
                })
    return rows

# -------------------------------
# 4. Parse all labels
# -------------------------------
train_data = parse_labels(TRAIN_LABELS_DIR, TRAIN_IMAGES_DIR, 'train')
valid_data = parse_labels(VALID_LABELS_DIR, VALID_IMAGES_DIR, 'valid')

# -------------------------------
# 5. Combine and save as CSV
# -------------------------------
all_data = train_data + valid_data
df = pd.DataFrame(all_data)
df.to_csv('dataset_labels.csv', index=False)
print("CSV file saved: dataset_labels.csv")


In [ ]:
import pandas as pd

df = pd.read_csv('/kaggle/working/dataset_labels.csv')
print(df.head())
print(df.info())
print(df['split'].value_counts())


In [ ]:
import os
import pandas as pd
import numpy as np
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model
from tqdm import tqdm

# -----------------------------
# 1. Load CSV
# -----------------------------
CSV_PATH = '/kaggle/working/dataset_labels.csv'
df = pd.read_csv(CSV_PATH)

# Optional: add class_name if missing
if 'class_name' not in df.columns:
    df['class_name'] = df['class_id'].apply(lambda x: 'debris' if x == 0 else f'class_{x}')

# -----------------------------
# 2. Prepare CNN model for feature extraction
# -----------------------------
base_model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')  # Global Average Pooling
feature_model = Model(inputs=base_model.input, outputs=base_model.output)

# -----------------------------
# 3. Define feature extraction function
# -----------------------------
def extract_image_features(img_path, target_size=(224, 224)):
    if not os.path.exists(img_path):
        return np.zeros((2048,), dtype=np.float32)  # fallback if image missing
    img = image.load_img(img_path, target_size=target_size)
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    features = feature_model.predict(x, verbose=0)
    return features.flatten()  # 2048-dim vector

# -----------------------------
# 4. Extract features for all images
# -----------------------------
# Remove duplicates if you want to extract features per unique image only
unique_images = df['image_path'].unique()
features_dict = {}

for img_path in tqdm(unique_images, desc="Extracting CNN features"):
    features_dict[img_path] = extract_image_features(img_path)

# -----------------------------
# 5. Add features to DataFrame
# -----------------------------
# Make a copy to avoid SettingWithCopyWarning
df = df.copy()
df['cnn_features'] = df['image_path'].apply(lambda x: features_dict.get(x, np.zeros((2048,), dtype=np.float32)))

# -----------------------------
# 6. Save DataFrame with features
# -----------------------------
OUTPUT_PATH = '/kaggle/working/dataset_with_features.pkl'
df.to_pickle(OUTPUT_PATH)
print(f"Saved DataFrame with CNN features: {OUTPUT_PATH}")


In [ ]:
import pandas as pd

df = pd.read_pickle('/kaggle/working/dataset_with_features.pkl')
print(df.head())


In [ ]:
train_df = df[df['split'] == 'train']
valid_df = df[df['split'] == 'valid']

X_train = list(train_df['cnn_features'])
X_valid = list(valid_df['cnn_features'])


In [ ]:
import numpy as np

X_train = np.array(X_train, dtype=np.float32)
X_valid = np.array(X_valid, dtype=np.float32)


In [ ]:
X_train_bbox = np.concatenate([X_train, train_df[['x_center','y_center','width','height']].values], axis=1)
X_valid_bbox = np.concatenate([X_valid, valid_df[['x_center','y_center','width','height']].values], axis=1)


In [ ]:
import pandas as pd
import numpy as np

# Load your dataframe (already has cnn_features)
df = pd.read_pickle('/kaggle/working/dataset_with_features.pkl')

# Add synthetic IDs
df = df.reset_index(drop=True)
df['object_id'] = df.index // 5        # Every 5 images = one pseudo object
df['frame_id'] = df.groupby('object_id').cumcount()

# Now simulate next-frame coordinates
df = df.sort_values(['object_id', 'frame_id']).reset_index(drop=True)
df['x_center_next'] = df.groupby('object_id')['x_center'].shift(-1)
df['y_center_next'] = df.groupby('object_id')['y_center'].shift(-1)

# Drop rows without next frame
df = df.dropna(subset=['x_center_next', 'y_center_next']).reset_index(drop=True)

print("✅ Added synthetic object and frame IDs + next-frame columns.")
print(df[['object_id','frame_id','x_center','x_center_next']].head())


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# Split train/valid
train_df = df[df['split'] == 'train']
valid_df = df[df['split'] == 'valid']

# Prepare CNN features + bounding box values
X_train = np.array(list(train_df['cnn_features']))
X_train = np.concatenate([X_train, train_df[['x_center','y_center','width','height']].values], axis=1)

X_valid = np.array(list(valid_df['cnn_features']))
X_valid = np.concatenate([X_valid, valid_df[['x_center','y_center','width','height']].values], axis=1)

# Targets (next position prediction)
y_train = train_df[['x_center_next','y_center_next']].values
y_valid = valid_df[['x_center_next','y_center_next']].values

print("✅ Data ready for model training.")
print("X_train:", X_train.shape, "y_train:", y_train.shape)

# ------------------------------
# 🧠 Define Neural Network model
# ------------------------------
model = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(2, activation='sigmoid')  # predicts next (x, y)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# ------------------------------
# ⚙️ Train model
# ------------------------------
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=15,
    batch_size=128,
    verbose=1
)

# ------------------------------
# 💾 Save trained model
# ------------------------------
model.save('/kaggle/working/trajectory_predictor.h5')

print("✅ Model trained and saved as trajectory_predictor.h5")


In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np

# --- Load model ---
model = tf.keras.models.load_model('/kaggle/working/trajectory_predictor.h5', compile=False)
model.compile(optimizer='adam',
              loss=tf.keras.losses.MeanSquaredError(),
              metrics=[tf.keras.metrics.MeanAbsoluteError()])

# --- Load dataset ---
df = pd.read_pickle('/kaggle/working/dataset_with_features.pkl')
print("Dataset shape:", df.shape)
print("Columns:", df.columns)

# --- Prepare feature arrays ---
cnn_features = np.stack(df['cnn_features'].values)  # (82796, 1280)
extra_features = df[['x_center', 'y_center', 'width', 'height']].values  # (82796, 4)

# Combine CNN + positional features
X = np.hstack([cnn_features, extra_features])  # (82796, 1284)
print("Combined input shape:", X.shape)

# --- Make predictions ---
predictions = model.predict(X)
print("Predictions shape:", predictions.shape)

# --- Add predictions back to DataFrame ---
if predictions.shape[1] == 2:
    df['pred_x'] = predictions[:, 0]
    df['pred_y'] = predictions[:, 1]
else:
    df['predicted'] = predictions[:, 0]

# --- Save results ---
df.to_csv('/kaggle/working/predicted_trajectories.csv', index=False)
print("✅ Predictions saved to /kaggle/working/predicted_trajectories.csv")

# --- (Optional) Inspect few rows ---
print(df[['x_center', 'y_center', 'pred_x', 'pred_y']].head())


In [ ]:
# =========================================================
# 1️⃣ Imports
# =========================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 2️⃣ Load datasets
# =========================================================
df = pd.read_pickle('/kaggle/working/dataset_with_features.pkl')
pred_df = pd.read_csv('/kaggle/working/predicted_trajectories.csv')

# Ensure synthetic object/frame IDs exist
if 'object_id' not in df.columns:
    df = df.reset_index(drop=True)
    df['object_id'] = df.index // 5
    df['frame_id'] = df.groupby('object_id').cumcount()

# Merge predictions
df = df.merge(pred_df[['pred_x','pred_y']], left_index=True, right_index=True)

# =========================================================
# 3️⃣ Compute errors
# =========================================================
df['x_error'] = df['x_center'] - df['pred_x']
df['y_error'] = df['y_center'] - df['pred_y']
df['error_magnitude'] = np.sqrt(df['x_error']**2 + df['y_error']**2)

# =========================================================
# 4️⃣ Scatter Plot: Actual vs Predicted
# =========================================================
plt.figure(figsize=(6,6))
plt.scatter(df['x_center'], df['y_center'], s=5, alpha=0.4, label='Actual')
plt.scatter(df['pred_x'], df['pred_y'], s=5, alpha=0.4, label='Predicted')
plt.title('Actual vs Predicted Object Positions')
plt.xlabel('X Center')
plt.ylabel('Y Center')
plt.legend()
plt.grid(True)
plt.show()

# =========================================================
# 5️⃣ Prediction Error Distribution
# =========================================================
plt.figure(figsize=(6,4))
sns.histplot(df['error_magnitude'], bins=50, kde=True)
plt.title('Prediction Error Magnitude Distribution')
plt.xlabel('Euclidean Error')
plt.ylabel('Count')
plt.show()

print(f"Mean Absolute Error (X): {df['x_error'].abs().mean():.4f}")
print(f"Mean Absolute Error (Y): {df['y_error'].abs().mean():.4f}")
print(f"Average Euclidean Error: {df['error_magnitude'].mean():.4f}")

# =========================================================
# 6️⃣ Sample Trajectory Simulation (first 5 objects)
# =========================================================
plt.figure(figsize=(8,6))
for obj_id in range(5):
    obj_df = df[df['object_id']==obj_id].sort_values('frame_id')
    plt.plot(obj_df['x_center'], obj_df['y_center'], marker='o', label=f'Actual {obj_id}')
    plt.plot(obj_df['pred_x'], obj_df['pred_y'], marker='x', linestyle='--', label=f'Predicted {obj_id}')

plt.title('Trajectory Simulation of Sample Objects')
plt.xlabel('X Center')
plt.ylabel('Y Center')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(df['x_center'], df['y_center'], s=5, alpha=0.4, label='Actual')
plt.scatter(df['pred_x'], df['pred_y'], s=5, alpha=0.4, label='Predicted')
plt.title('Actual vs Predicted Object Positions')
plt.xlabel('X Center')
plt.ylabel('Y Center')
plt.legend()
plt.grid(True)
plt.savefig('actual_vs_predicted_300dpi.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
import seaborn as sns
sns.histplot(df['error'], bins=50, kde=True)
plt.title('Prediction Error Magnitude Distribution')
plt.xlabel('Euclidean Error')
plt.ylabel('Count')
plt.savefig('error_magnitude_distribution_300dpi.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
for obj_id in range(5):
    obj_df = df[df['object_id']==obj_id].sort_values('frame_id')
    plt.plot(obj_df['x_center'], obj_df['y_center'], marker='o', label=f'Actual {obj_id}')
    plt.plot(obj_df['pred_x'], obj_df['pred_y'], marker='x', linestyle='--', label=f'Predicted {obj_id}')
plt.title('Trajectory Simulation of Sample Objects')
plt.xlabel('X Center')
plt.ylabel('Y Center')
plt.legend()
plt.grid(True)
plt.savefig('trajectory_simulation_300dpi.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Compute errors (run this BEFORE any plotting or referencing errors)
df['x_error'] = df['x_center'] - df['pred_x']
df['y_error'] = df['y_center'] - df['pred_y']
df['error_magnitude'] = np.sqrt(df['x_error']**2 + df['y_error']**2)



In [ ]:
mae_x = df['x_error'].abs().mean()
mae_y = df['y_error'].abs().mean()
mse = ((df['x_error']**2 + df['y_error']**2)/2).mean()
rmse = np.sqrt(mse)
avg_euclidean_error = df['error_magnitude'].mean()

print(f"MAE X: {mae_x:.4f}, MAE Y: {mae_y:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"Average Euclidean Error: {avg_euclidean_error:.4f}")


In [ ]:
import numpy as np

# Thresholds to test
thresholds = [0.3,0.4,0.5]

for tol in thresholds:
    # Compute Euclidean distance between predicted and actual positions
    df['error'] = np.sqrt((df['x_center'] - df['pred_x'])**2 + (df['y_center'] - df['pred_y'])**2)
    
    # Count correct and wrong predictions
    correct = (df['error'] <= tol).sum()
    wrong = (df['error'] > tol).sum()
    total = len(df)
    
    # Percentage
    correct_pct = (correct / total) * 100
    wrong_pct = (wrong / total) * 100
    
    print(f"Threshold: {tol}")
    print(f"Correct predictions: {correct} ({correct_pct:.2f}%)")
    print(f"Wrong predictions: {wrong} ({wrong_pct:.2f}%)\n")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------
# 1️⃣ Load predictions dataset
# -------------------------------
df = pd.read_csv('/kaggle/working/predicted_trajectories.csv')  # columns: x_center, y_center, pred_x, pred_y

# -------------------------------
# 2️⃣ Compute errors
# -------------------------------
df['error'] = np.sqrt((df['x_center'] - df['pred_x'])**2 + (df['y_center'] - df['pred_y'])**2)

mae_x = np.mean(np.abs(df['x_center'] - df['pred_x']))
mae_y = np.mean(np.abs(df['y_center'] - df['pred_y']))
mse = np.mean(df['error']**2)
rmse = np.sqrt(mse)
avg_euc_error = np.mean(df['error'])

print(f"MAE X: {mae_x:.4f}, MAE Y: {mae_y:.4f}")
print(f"MSE: {mse:.4f}, RMSE: {rmse:.4f}")
print(f"Average Euclidean Error: {avg_euc_error:.4f}")

# -------------------------------
# 3️⃣ Threshold-based accuracy
# -------------------------------
threshold = 0.5
correct = (df['error'] <= threshold).sum()
wrong = (df['error'] > threshold).sum()
total = len(df)

print(f"Threshold: {threshold}")
print(f"Total predictions: {total}")
print(f"Correct predictions: {correct} ({correct/total*100:.2f}%)")
print(f"Wrong predictions: {wrong} ({wrong/total*100:.2f}%)")

# -------------------------------
# 4️⃣ Plot Euclidean error distribution
# -------------------------------
plt.figure(figsize=(10,5))
plt.hist(df['error'], bins=50, color='skyblue', edgecolor='black')
plt.axvline(threshold, color='red', linestyle='--', label=f'Threshold {threshold}')
plt.title('YOLO: Distribution of Euclidean Errors')
plt.xlabel('Error (Euclidean distance)')
plt.ylabel('Count')
plt.legend()
plt.grid(True)
plt.show()

# -------------------------------
# 5️⃣ Plot correct vs wrong predictions
# -------------------------------
plt.figure(figsize=(6,5))
plt.bar(['Correct', 'Wrong'], [correct, wrong], color=['green','red'])
plt.title(f'YOLO: Predictions at Threshold {threshold} ({correct/total*100:.2f}% accuracy)')
plt.ylabel('Number of predictions')
plt.show()

# -------------------------------
# 6️⃣ Sample trajectory curves (first 5 samples)
# -------------------------------
plt.figure(figsize=(10,8))
for i in range(5):
    plt.plot([df['x_center'].iloc[i], df['pred_x'].iloc[i]],
             [df['y_center'].iloc[i], df['pred_y'].iloc[i]],
             marker='o', linestyle='--', label=f'Sample {i}')

plt.title('YOLO: Actual vs Predicted Trajectories (Sample 5 points)')
plt.xlabel('X coordinate')
plt.ylabel('Y coordinate')
plt.legend()
plt.grid(True)
plt.show()

# -------------------------------
# 7️⃣ Plot MAE/MSE/RMSE
# -------------------------------
plt.figure(figsize=(6,5))
plt.bar(['MAE_X', 'MAE_Y', 'RMSE'], [mae_x, mae_y, rmse], color=['blue','orange','purple'])
plt.title('YOLO: Error Metrics')
plt.ylabel('Error value')
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1️⃣ Load predictions dataset
df = pd.read_csv('/kaggle/working/predicted_trajectories.csv')  # columns: x_center, y_center, pred_x, pred_y

# 2️⃣ Compute errors
df['error'] = np.sqrt((df['x_center'] - df['pred_x'])**2 + (df['y_center'] - df['pred_y'])**2)
mae_x = np.mean(np.abs(df['x_center'] - df['pred_x']))
mae_y = np.mean(np.abs(df['y_center'] - df['pred_y']))
mse = np.mean(df['error']**2)
rmse = np.sqrt(mse)
avg_euc_error = np.mean(df['error'])

# 3️⃣ Threshold-based accuracy
threshold = 0.5
correct = (df['error'] <= threshold).sum()
wrong = (df['error'] > threshold).sum()
total = len(df)

# 4️⃣ Plot Euclidean error distribution
plt.figure(figsize=(10,5))
plt.hist(df['error'], bins=50, color='skyblue', edgecolor='black')
plt.axvline(threshold, color='red', linestyle='--', label=f'Threshold {threshold}')
plt.title('YOLO: Distribution of Euclidean Errors')
plt.xlabel('Error (Euclidean distance)')
plt.ylabel('Count')
plt.legend()
plt.grid(True)
plt.savefig('euclidean_error_distribution_300dpi.png', dpi=300, bbox_inches='tight')
plt.show()

# 5️⃣ Plot correct vs wrong predictions
plt.figure(figsize=(6,5))
plt.bar(['Correct', 'Wrong'], [correct, wrong], color=['green','red'])
plt.title(f'YOLO: Predictions at Threshold {threshold} ({correct/total*100:.2f}% accuracy)')
plt.ylabel('Number of predictions')
plt.savefig('correct_vs_wrong_predictions_300dpi.png', dpi=300, bbox_inches='tight')
plt.show()

# 6️⃣ Sample trajectory curves (first 5 samples)
plt.figure(figsize=(10,8))
for i in range(5):
    plt.plot([df['x_center'].iloc[i], df['pred_x'].iloc[i]],
             [df['y_center'].iloc[i], df['pred_y'].iloc[i]],
             marker='o', linestyle='--', label=f'Sample {i}')
plt.title('YOLO: Actual vs Predicted Trajectories (Sample 5 points)')
plt.xlabel('X coordinate')
plt.ylabel('Y coordinate')
plt.legend()
plt.grid(True)
plt.savefig('trajectory_curve_samples_300dpi.png', dpi=300, bbox_inches='tight')
plt.show()

# 7️⃣ Plot MAE/MSE/RMSE
plt.figure(figsize=(6,5))
plt.bar(['MAE_X', 'MAE_Y', 'RMSE'], [mae_x, mae_y, rmse], color=['blue','orange','purple'])
plt.title('YOLO: Error Metrics')
plt.ylabel('Error value')
plt.savefig('error_metrics_300dpi.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# =========================================
# 1️⃣ Imports
# =========================================
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# =========================================
# 2️⃣ Load dataset with CNN features
# =========================================
df = pd.read_pickle('/kaggle/working/dataset_with_features.pkl')

# Add synthetic object_id and frame_id
df = df.reset_index(drop=True)
df['object_id'] = df.index // 5
df['frame_id'] = df.groupby('object_id').cumcount()

# Sort and generate next-frame coordinates
df = df.sort_values(['object_id','frame_id']).reset_index(drop=True)
df['x_center_next'] = df.groupby('object_id')['x_center'].shift(-1)
df['y_center_next'] = df.groupby('object_id')['y_center'].shift(-1)
df = df.dropna(subset=['x_center_next','y_center_next']).reset_index(drop=True)

print("Dataset ready. Example:")
print(df[['object_id','frame_id','x_center','x_center_next']].head())

# =========================================
# 3️⃣ Sequence creation for LSTM
# =========================================
SEQ_LEN = 1  # can increase to 2 or 3 if objects have more frames
X_sequences = []
y_sequences = []

for obj_id, obj_df in df.groupby('object_id'):
    obj_df = obj_df.reset_index(drop=True)
    num_frames = len(obj_df)
    
    if num_frames <= SEQ_LEN:
        continue  # skip short sequences
    
    cnn_feats = np.stack(obj_df['cnn_features'].values)
    coords = obj_df[['x_center','y_center']].values
    features = np.concatenate([coords, cnn_feats], axis=1)
    
    for i in range(num_frames - SEQ_LEN):
        X_sequences.append(features[i:i+SEQ_LEN])
        y_sequences.append(coords[i+SEQ_LEN])

X = np.array(X_sequences)
y = np.array(y_sequences)

print(f"Number of sequences: {X.shape[0]}")
print(f"Input shape: {X.shape}")
print(f"Target shape: {y.shape}")

# =========================================
# 4️⃣ Train-Test split
# =========================================
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.1, random_state=42)

print("Train shape:", X_train.shape, y_train.shape)
print("Valid shape:", X_valid.shape, y_valid.shape)

# =========================================
# 5️⃣ LSTM model
# =========================================
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQ_LEN, X.shape[2])),
    tf.keras.layers.LSTM(256, return_sequences=False),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(2)  # x_center_next, y_center_next
])

model.compile(optimizer='adam', 
              loss=tf.keras.losses.MeanSquaredError(),
              metrics=[tf.keras.metrics.MeanAbsoluteError()])

model.summary()

# =========================================
# 6️⃣ Train model
# =========================================
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=15,
    batch_size=32
)

# =========================================
# 7️⃣ Make predictions
# =========================================
predictions = model.predict(X_valid)
print("Predictions shape:", predictions.shape)

# Save predictions with coordinates
valid_df_seq = df.iloc[:len(y_valid)].copy()
valid_df_seq['pred_x'] = predictions[:,0]
valid_df_seq['pred_y'] = predictions[:,1]
valid_df_seq[['x_center','y_center','pred_x','pred_y']].to_csv('/kaggle/working/predicted_trajectories.csv', index=False)
print("✅ Predictions saved to /kaggle/working/predicted_trajectories.csv")

# =========================================
# 8️⃣ Simple visualization for first 5 objects
# =========================================
plt.figure(figsize=(10,6))
for obj_id in range(5):
    obj_df = valid_df_seq[valid_df_seq['object_id']==obj_id]
    if obj_df.empty:
        continue
    plt.plot(obj_df['x_center'], obj_df['y_center'], marker='o', label=f'Actual {obj_id}')
    plt.plot(obj_df['pred_x'], obj_df['pred_y'], marker='x', linestyle='--', label=f'Predicted {obj_id}')

plt.xlabel("x_center")
plt.ylabel("y_center")
plt.title("Trajectory Predictions (first 5 objects)")
plt.legend()
plt.show()

# =========================================
# 9️⃣ Compute MAE, MSE, RMSE, Euclidean Error
# =========================================
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae_x = mean_absolute_error(y_valid[:,0], predictions[:,0])
mae_y = mean_absolute_error(y_valid[:,1], predictions[:,1])
mse = mean_squared_error(y_valid, predictions)
rmse = np.sqrt(mse)
euclidean = np.mean(np.sqrt(np.sum((y_valid - predictions)**2, axis=1)))

print(f"MAE X: {mae_x:.4f}, MAE Y: {mae_y:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"Average Euclidean Error: {euclidean:.4f}")


In [ ]:
import pandas as pd
import numpy as np

# Load original dataset with synthetic next-frame coords
df = pd.read_pickle('/kaggle/working/dataset_with_features.pkl')
df = df.reset_index(drop=True)
df['object_id'] = df.index // 5
df['frame_id'] = df.groupby('object_id').cumcount()
df = df.sort_values(['object_id', 'frame_id']).reset_index(drop=True)
df['x_center_next'] = df.groupby('object_id')['x_center'].shift(-1)
df['y_center_next'] = df.groupby('object_id')['y_center'].shift(-1)
df = df.dropna(subset=['x_center_next','y_center_next']).reset_index(drop=True)

# Load prediction CSV
df_pred = pd.read_csv('/kaggle/working/predicted_trajectories.csv')

# Ensure we have object_id and frame_id in predictions (or use image_path to merge)
# If predictions were generated in order, we can assign sequential IDs
df_pred = df_pred.reset_index(drop=True)
df_pred['object_id'] = df_pred.index // 5
df_pred['frame_id'] = df_pred.groupby('object_id').cumcount()

# Merge on object_id + frame_id
df_merged = pd.merge(df, df_pred[['object_id','frame_id','pred_x','pred_y']],
                     on=['object_id','frame_id'], how='inner')

# Compute Euclidean error
df_merged['error'] = np.sqrt((df_merged['x_center_next'] - df_merged['pred_x'])**2 +
                             (df_merged['y_center_next'] - df_merged['pred_y'])**2)

# Evaluate at multiple thresholds
thresholds = [0.01,0.05,0.1,0.2,0.3,0.4,0.5]
print("Threshold-based prediction analysis:")
for t in thresholds:
    correct = (df_merged['error'] < t).sum()
    wrong = (df_merged['error'] >= t).sum()
    total = len(df_merged)
    acc = correct / total * 100
    print(f"\nThreshold: {t}")
    print(f"Correct predictions: {correct} ({acc:.2f}%)")
    print(f"Wrong predictions: {wrong} ({100-acc:.2f}%)")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------
# 1️⃣ Load predictions dataset
# -------------------------------
df = pd.read_csv('/kaggle/working/predicted_trajectories.csv')  # columns: x_center, y_center, pred_x, pred_y

# -------------------------------
# 2️⃣ Add synthetic object/frame IDs
# -------------------------------
df = df.reset_index(drop=True)
df['object_id'] = df.index // 5
df['frame_id'] = df.groupby('object_id').cumcount()

# Next-frame actual positions
df = df.sort_values(['object_id', 'frame_id']).reset_index(drop=True)
df['x_center_next'] = df.groupby('object_id')['x_center'].shift(-1)
df['y_center_next'] = df.groupby('object_id')['y_center'].shift(-1)
df = df.dropna(subset=['x_center_next', 'y_center_next']).reset_index(drop=True)

# -------------------------------
# 3️⃣ Compute errors
# -------------------------------
df['error'] = np.sqrt((df['x_center_next'] - df['pred_x'])**2 + 
                      (df['y_center_next'] - df['pred_y'])**2)

mae_x = np.mean(np.abs(df['x_center_next'] - df['pred_x']))
mae_y = np.mean(np.abs(df['y_center_next'] - df['pred_y']))
mse = np.mean(df['error']**2)
rmse = np.sqrt(mse)
avg_euc_error = np.mean(df['error'])

print(f"MAE X: {mae_x:.4f}, MAE Y: {mae_y:.4f}")
print(f"MSE: {mse:.4f}, RMSE: {rmse:.4f}")
print(f"Average Euclidean Error: {avg_euc_error:.4f}")

# -------------------------------
# 4️⃣ Threshold-based accuracy for 0.5
# -------------------------------
threshold = 0.5
correct = (df['error'] <= threshold).sum()
wrong = (df['error'] > threshold).sum()
total = len(df)

print(f"Threshold: {threshold}")
print(f"Total predictions: {total}")
print(f"Correct predictions: {correct} ({correct/total*100:.2f}%)")
print(f"Wrong predictions: {wrong} ({wrong/total*100:.2f}%)")

# -------------------------------
# 5️⃣ Plot Euclidean error distribution
# -------------------------------
plt.figure(figsize=(10,5))
plt.hist(df['error'], bins=50, color='skyblue', edgecolor='black')
plt.axvline(threshold, color='red', linestyle='--', label=f'Threshold {threshold}')
plt.title('Distribution of Euclidean Errors')
plt.xlabel('Error (Euclidean distance)')
plt.ylabel('Count')
plt.legend()
plt.grid(True)
plt.show()

# -------------------------------
# 6️⃣ Plot precision/accuracy bar
# -------------------------------
plt.figure(figsize=(6,5))
plt.bar(['Correct', 'Wrong'], [correct, wrong], color=['green','red'])
plt.title(f'Predictions at Threshold {threshold} ({correct/total*100:.2f}% accuracy)')
plt.ylabel('Number of predictions')
plt.show()

# -------------------------------
# 7️⃣ Sample trajectory curves (first 5 objects)
# -------------------------------
sample_ids = df['object_id'].unique()[:5]
plt.figure(figsize=(10,8))
for obj_id in sample_ids:
    obj_df = df[df['object_id']==obj_id]
    plt.plot(obj_df['x_center'], obj_df['y_center'], marker='o', label=f'Actual {obj_id}')
    plt.plot(obj_df['pred_x'], obj_df['pred_y'], marker='x', linestyle='--', label=f'Predicted {obj_id}')

plt.title('Actual vs Predicted Trajectories')
plt.xlabel('X coordinate')
plt.ylabel('Y coordinate')
plt.legend()
plt.grid(True)
plt.show()

# -------------------------------
# 8️⃣ MAE/MSE error bars
# -------------------------------
plt.figure(figsize=(6,5))
plt.bar(['MAE_X', 'MAE_Y', 'RMSE'], [mae_x, mae_y, rmse], color=['blue','orange','purple'])
plt.title('Error Metrics')
plt.ylabel('Error value')
plt.show()
